In [8]:
from PIL import Image
import cv2

import matplotlib.pyplot as plt
import numpy as np

from glob import glob
import time
import json
from tqdm import tqdm

import torch
from transformers import AutoProcessor, AutoModelForCausalLM

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

## 1. 工具函数

In [2]:
def xyxy_to_xywh(coords):
    x1 = coords[0]
    y1 = coords[1]
    x2 = coords[2]
    y2 = coords[3]
    
    w = x2 - x1
    h = y2 - y1
    return [x1, y1, w, h]

def xyxy_get_area(coords):
    x1 = coords[0]
    y1 = coords[1]
    x2 = coords[2]
    y2 = coords[3]

    w = x2 - x1
    h = y2 - y1
    return w*h

def show_img(img, bboxes = None, tag = None, captions = None):
    '''
        Function: used to show image and corresponding bounding boxes.
    '''
    fig, ax = plt.subplots()
    
    ax.imshow(img)
    
    if bboxes is not None:
        for i, bbox in enumerate(bboxes):
            x, y, w, h = bbox
            rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            
            if tag is not None:
                ax.text(x, y-8, str(i+1), color='r', fontsize=11, weight='bold')
            if captions is not None:
                ax.text(x+20, y-8, captions[i], color='r', fontsize=11, weight='bold')
    plt.axis("off")
    plt.show()

In [3]:
def load_florence(model_id = "microsoft/Florence-2-large"):
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        trust_remote_code=True,
        torch_dtype=torch_dtype 
    ).eval().to(device)
    
    processor = AutoProcessor.from_pretrained(
        model_id, 
        trust_remote_code=True
    )
    return model, processor

In [4]:
def florence_processor(model, processor, img, caption):
    task_prompt = "<OPEN_VOCABULARY_DETECTION>"
    prompt = task_prompt + caption
    
    # 1. 利用image和caption制作输入
    inputs = processor(text=prompt, images=img, return_tensors="pt").to(device, torch_dtype)

    # 2. 得到输出内容
    start_time = time.time()
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024, # 影响生成阶段最大的新token数量。这个参数的设置，直接影响输出的长度。
        num_beams=3, # 参数控制了生成阶段使用的beam搜索数量。增加beam数量可以提高搜索的广度，从而提高输出的多样性，但同时也增加了计算量。
        return_dict_in_generate=True,
        output_scores=True,
    )
    inference_time = time.time() - start_time

    # 3. 将输出内容进行解码
    generated_text = processor.batch_decode(generated_ids.sequences, skip_special_tokens=False)[0]

    # 4. 分析输出的内容
    # prediction, scores, beam_indices = generated_ids.sequences, generated_ids.scores, generated_ids.beam_indices
    transition_beam_scores = model.compute_transition_scores(
        sequences = generated_ids.sequences,
        scores = generated_ids.scores,
        beam_indices = generated_ids.beam_indices,
    )

    # 5. 得到结果
    parsed_answer = processor.post_process_generation(
        sequence = generated_ids.sequences[0], 
        transition_beam_score = transition_beam_scores[0],
        task="<OD>", 
        image_size=(img.width, img.height)
    )

    bboxes = parsed_answer["<OD>"]["bboxes"]
    labels = parsed_answer["<OD>"]["labels"]
    scores = parsed_answer["<OD>"]["scores"]
    areas = [xyxy_get_area(bbox) for bbox in bboxes]

    result = []
    for bbox, area, label, score in zip(bboxes, areas, labels, scores):
        result.append({
            "bbox":xyxy_to_xywh(np.array(bbox).astype(np.int64).tolist()),
            "area":xyxy_get_area(bbox),
            "label":label,
            "score":score
        })

    return result, img.size, inference_time

## 2. Florence模型推断

In [30]:
model_id = "microsoft/Florence-2-large" # microsoft/Florence-2-base
model, processor = load_florence(model_id=model_id)

caption = "excavators" # florence是对单复数敏感的。cylinder 只会获取一个bbox；cylinders 会获取所有的bbox.

In [31]:
img_path_list = glob("./imgs/dataset/5. vehicle/*.jpg")
print(f"The quantity of images: {len(img_path_list)}")

The quantity of images: 837


In [32]:
img_outcome = {}
for i, img_path in tqdm(enumerate(img_path_list), total = len(img_path_list)):
    img = Image.open(img_path).convert("RGB")

    outcome, image_size, inference_time = florence_processor(model=model, processor=processor, img = img, caption=caption)

    img_outcome[i] = {
        "img_path":img_path,
        "img_size":image_size, #[width, height]
        "inference_time":inference_time,
        "result":outcome
    }

100%|████████████████████████████████████████████████████████████████████████████████| 837/837 [18:31<00:00,  1.33s/it]


In [34]:
with open(f"./results/florence_{5}.json", "w") as fp:
    fp.write(json.dumps(img_outcome))

## 3. 测试代码

In [20]:
# model = AutoModelForCausalLM.from_pretrained(
#     model_id, 
#     trust_remote_code=True,
#     torch_dtype=torch_dtype 
# ).eval().to(device)

# processor = AutoProcessor.from_pretrained(
#     model_id, 
#     trust_remote_code=True
# )

In [21]:
# task_prompt = "<OPEN_VOCABULARY_DETECTION>"
# text_input = "cylinders"
# prompt = task_prompt + text_input

# img = Image.open("./imgs/testing.jpg")

# inputs = processor(
#     text=prompt, 
#     images=img, 
#     return_tensors="pt"
# ).to(device, torch_dtype)

In [22]:
# generated_ids = model.generate(
#     input_ids=inputs["input_ids"],
#     pixel_values=inputs["pixel_values"],
#     max_new_tokens=1024, # 影响生成阶段最大的新token数量。这个参数的设置，直接影响输出的长度。
#     num_beams=3, # 参数控制了生成阶段使用的beam搜索数量。增加beam数量可以提高搜索的广度，从而提高输出的多样性，但同时也增加了计算量。
#     return_dict_in_generate=True,
#     output_scores=True,
# )

# generated_text = processor.batch_decode(generated_ids.sequences, skip_special_tokens=False)[0]

# prediction, scores, beam_indices = generated_ids.sequences, generated_ids.scores, generated_ids.beam_indices

# transition_beam_scores = model.compute_transition_scores(
#     sequences = prediction,
#     scores = scores,
#     beam_indices = beam_indices,
# )

# parsed_answer = processor.post_process_generation(
#     sequence = generated_ids.sequences[0], 
#     transition_beam_score = transition_beam_scores[0],
#     task="<OD>", 
#     image_size=(img.width, img.height)
# )